# P5-2 · GĐ 2 · Diffusion cho ảnh

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers

## Lý thuyết

In [ ]:
import torch
from diffusers import UNet2DModel
model = UNet2DModel(sample_size=32, in_channels=1, out_channels=1, layers_per_block=2,
                    block_out_channels=(32, 64, 128),
                    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
                    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"))
x_t, t = torch.randn(4, 1, 32, 32), torch.randint(0, 1000, (4,))   # batch 4 ảnh nhiễu và bước t
eps_hat = model(x_t, t).sample          # x_t: (B,1,32,32), t: (B,) số nguyên -> (B,1,32,32)
print(eps_hat.shape, f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu tham số")

torch.Size([4, 1, 32, 32]) 4.02 triệu tham số


In [ ]:
import torch
T = 1000
alpha_bars = torch.cumprod(1 - torch.linspace(1e-4, 0.02, T), dim=0)   # lịch nhiễu như GĐ1

@torch.no_grad()
def sample_ddim(model, n, steps=50):
    x = torch.randn(n, 1, 32, 32)
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    for i, t in enumerate(ts):
        eps = model(x, torch.full((n,), t)).sample
        ab = alpha_bars[t]
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if i == len(ts) - 1:
            return x0.clamp(-1, 1)
        ab_s = alpha_bars[ts[i + 1]]
        x = ab_s.sqrt() * x0 + (1 - ab_s).sqrt() * eps

imgs = sample_ddim(model.eval(), n=2, steps=10)    # model ở ô trên chưa train nên ảnh chỉ là nhiễu; ở đây chỉ kiểm tra chạy đúng
imgs.shape, imgs.min().item(), imgs.max().item()

(torch.Size([2, 1, 32, 32]), -1.0, 1.0)


In [ ]:
import copy, torch
ema = copy.deepcopy(model).eval().requires_grad_(False)

@torch.no_grad()
def update_ema(ema, model, decay=0.999):
    for pe, p in zip(ema.parameters(), model.parameters()):
        pe.mul_(decay).add_(p, alpha=1 - decay)
# gọi update_ema(ema, model) sau mỗi opt.step(); sinh ảnh bằng ema, không phải model

# thử: đổi trọng số model rồi cập nhật EMA 1 lần và 1000 lần
w = next(model.parameters()); w_ema = next(ema.parameters())
with torch.no_grad(): w.add_(1.0)                     # giả sử một bước train làm trọng số tăng 1
update_ema(ema, model)
print("sau 1 lần:", round((w - w_ema).abs().mean().item(), 4))      # EMA mới đi được 0,1% quãng đường
for _ in range(999): update_ema(ema, model)
print("sau 1000 lần:", round((w - w_ema).abs().mean().item(), 4))   # còn khoảng e^-1 ≈ 37%

sau 1 lần: 0.999
sau 1000 lần: 0.3677
